# Tornado CNN inference and evaluation

Set `CONFIG_PATH` below to the training YAML. The notebook uses its test array paths, model configuration, checkpoint path, and probability threshold.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

SCRIPT_DIR = Path.cwd() if Path('mod_model.py').exists() else Path('scripts')
sys.path.insert(0, str(SCRIPT_DIR.resolve()))
from mod_model import build_model
from main import read_array, read_labels

CONFIG_PATH = SCRIPT_DIR / 'tornado.yaml'
with open(CONFIG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint_path = Path(cfg['training']['output_path'])
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model_cfg = checkpoint.get('model', cfg['model'])
model = build_model(model_cfg).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()


In [ ]:
# Features are read from NetCDF; labels are read from a one-column CSV.
x = np.asarray(read_array(cfg['inference']['test_x']), dtype=np.float32)
y_true = np.asarray(read_labels(cfg['inference']['test_y'])).reshape(-1).astype(np.int64)
if x.ndim != 4 or x.shape[0] != len(y_true):
    raise ValueError(f'Expected X [nb,nx,ny,nc] and matching Y [nb], got {x.shape}, {y_true.shape}')
if not np.isin(y_true, [0, 1]).all():
    raise ValueError('Test labels must be binary (0 or 1)')

batch_size = int(cfg['training'].get('batch_size', 32))
probability_parts = []
with torch.no_grad():
    for start in range(0, len(x), batch_size):
        xb = torch.from_numpy(x[start:start + batch_size]).to(device)
        probability_parts.append(torch.sigmoid(model(xb)).cpu().numpy())
y_probability = np.concatenate(probability_parts) if probability_parts else np.empty((0,), dtype=np.float32)
threshold = float(checkpoint.get('threshold', cfg.get('inference', {}).get('threshold', 0.5)))
y_pred = (y_probability >= threshold).astype(np.int64)

scores = {
    'Precision': precision_score(y_true, y_pred, zero_division=0),
    'Recall': recall_score(y_true, y_pred, zero_division=0),
    'F1': f1_score(y_true, y_pred, zero_division=0),
}
print(f'Threshold: {threshold:.3f}; test samples: {len(y_true)}')
print(scores)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(list(scores), list(scores.values()), color=['#4c78a8', '#f58518', '#54a24b'])
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Score')
axes[0].set_title('Test classification metrics')
axes[0].grid(axis='y', alpha=0.25)
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=[0, 1]), display_labels=[0, 1]).plot(ax=axes[1], colorbar=False)
axes[1].set_title('Test confusion matrix')
plt.tight_layout()
plt.show()


In [ ]:
# Per-sample probabilities can be inspected or saved for downstream use.
probability_output = Path('test_probabilities.npy')
np.save(probability_output, y_probability)
print(f'Saved probabilities to {probability_output.resolve()}')